# 🛟 실행 복구 — 런타임이 끊겨도 결과는 Drive에 있다

**MedKOS / `notebooks/recover_run.ipynb`**

Colab 런타임이 끊겨서 마지막 셀(`result.json` 저장 + `registry.jsonl` 기록)을 못 돌렸을 때,
**이미 Drive에 저장된 산출물만으로 복구**합니다. 학습을 다시 하지 않습니다.

이게 가능한 이유 — 실험 노트북이 매 단계마다 Drive에 즉시 기록하기 때문입니다:

```
runs/<run_id>/
├── config.json          설정
├── cohort_stats.json    코호트 규모
├── evaluation.json      ★ 모든 지표·부트스트랩·판정
├── arms/<arm>/probs.npy ★ 예측 확률 (재학습 없이 재평가 가능)
└── result.json          ← 이것만 없으면 여기서 다시 만든다
```

**GPU 필요 없습니다.** 런타임 유형은 CPU로 두셔도 됩니다.

In [ ]:
# CELL 1 — Drive 마운트 + 실행 목록 (어느 run이 미완성인지 확인)
import os, json, time

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"

PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
RUNS = os.path.join(PROJECT, "runs")
REG = os.path.join(PROJECT, "registry.jsonl")

logged = set()
if os.path.exists(REG):
    for line in open(REG):
        try:
            logged.add(json.loads(line).get("run_id"))
        except Exception:
            pass

rows = []
for rid in sorted(os.listdir(RUNS)) if os.path.exists(RUNS) else []:
    d = os.path.join(RUNS, rid)
    if not os.path.isdir(d):
        continue
    have = {f.replace(".json", "") for f in os.listdir(d) if f.endswith(".json")}
    arms = sorted(os.listdir(os.path.join(d, "arms"))) if os.path.isdir(os.path.join(d, "arms")) else []
    rows.append((rid, "evaluation" in have, "result" in have, rid in logged, arms))

print(f"{'RUN_ID':<34}{'eval':>6}{'result':>8}{'대장':>6}  arms")
print("=" * 84)
for rid, ev, rs, lg, arms in rows:
    print(f"{rid:<34}{'✅' if ev else '—':>6}{'✅' if rs else '❌':>8}"
          f"{'✅' if lg else '❌':>6}  {','.join(arms)}")
print("=" * 84)
todo = [r for r in rows if r[1] and not (r[2] and r[3])]
print(f"\n복구 대상(평가는 끝났는데 result/대장이 없는 run): {[r[0] for r in todo] or '없음'}")

### CELL 2 — 복구 실행

`RUN_ID`를 비워두면 **복구 대상 중 가장 최근 것**을 자동으로 고릅니다.

In [ ]:
# CELL 2 — result.json 재구성 + registry.jsonl 기록 + 다운로드
RUN_ID = ""          # 비우면 자동 선택. 특정 run을 지정하려면 위 목록에서 복사

rid = RUN_ID or (todo[-1][0] if todo else (rows[-1][0] if rows else None))
if not rid:
    raise SystemExit("복구할 run이 없습니다.")
d = os.path.join(RUNS, rid)
print(f"복구 대상: {rid}\n  {d}\n")

def load(name, default=None):
    p = os.path.join(d, f"{name}.json")
    return json.load(open(p)) if os.path.exists(p) else default

cfg = load("config", {}) or {}
ev = load("evaluation")
coh = load("cohort_stats", {}) or {}
if ev is None:
    raise SystemExit("❌ evaluation.json이 없습니다 — 평가 셀까지는 돌려야 복구됩니다.")

arms = ev.get("arms", {})
boots = ev.get("bootstrap", {})
verdict = ev.get("verdict", "")
exp_id = cfg.get("exp", rid.split("_", 1)[-1])

# 주가설 찾기: B3R-B3 (실험1′) 또는 B3P_minus_B3 (실험2) 형식 모두 대응
key = next((k for k in boots if k.replace(" ", "").upper() in ("B3R-B3", "B3P-B3")), None)
main = boots.get(key) if key else ev.get("B3P_minus_B3")
delta = main.get("delta") if main else None
ci = main.get("ci") if main else None
best_arm = max(arms, key=lambda a: arms[a].get("macro_f1", 0)) if arms else None

summary = (f"{key or 'main'} Δ{delta:+.4f} CI[{ci[0]:+.4f},{ci[1]:+.4f}] "
           f"n_pat={coh.get('n_patients','?')} → {verdict.split(' —')[0]}"
           if delta is not None and ci else verdict[:80])

result = {
    "week": 1, "exp_id": exp_id, "quest": cfg.get("quest", "ailab-2026-0015"),
    "task": cfg.get("hypothesis", exp_id), "split": "inter", "metric": "macro_f1",
    "value": round(arms[best_arm]["macro_f1"], 4) if best_arm else None,
    "passed": bool(ci and (ci[0] > 0) and delta and delta > 0
                   and not ev.get("collapsed_arms") and not ev.get("collapsed")),
    "date": time.strftime("%Y-%m-%d", time.localtime(os.path.getmtime(
        os.path.join(d, "evaluation.json")))),
    "n_patients": coh.get("n_patients"),
    "n_test_patients": ev.get("n_patients_with_ectopy"),
    "n_beats": coh.get("n_beats"),
    "arms": {a: v.get("macro_f1") for a, v in arms.items()},
    "bootstrap": boots or {"main": main},
    "ci_halfwidth": round((ci[1] - ci[0]) / 2, 4) if ci else None,
    "verdict": verdict, "summary": summary,
    "recovered": True, "run_id": rid,
    "config": {k: cfg.get(k) for k in ("n_patients", "minutes", "n_seeds", "epochs", "quick")},
}

p = os.path.join(d, "result.json")
json.dump(result, open(p, "w"), ensure_ascii=False, indent=2, default=str)
print("✅ result.json 재생성:", p)

if rid not in logged:
    line = {"run_id": rid, "exp_id": exp_id, "date": result["date"],
            "metric": "macro_f1", "value": result["value"], "passed": result["passed"],
            "summary": summary, "dir": d}
    with open(REG, "a") as f:
        f.write(json.dumps(line, ensure_ascii=False) + "\n")
    print("✅ registry.jsonl 에 기록 추가")
else:
    print("ℹ️ 이미 대장에 있음 — 중복 기록 안 함")

print("\n" + json.dumps({k: v for k, v in result.items() if k != "bootstrap"},
                        ensure_ascii=False, indent=2)[:1200])
print(f"""
────────────────────────────────────────────────────────────────
repo에 박기 — 아래 파일을 내려받아 repo 루트에 두고 실행

  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp1_icentia_rhythm_scale.ipynb \\
      --quest ailab-2026-0015 --step "exp1p-icentia-scale" \\
      --note "{summary}"
────────────────────────────────────────────────────────────────""")

try:
    from google.colab import files
    files.download(p)                      # 브라우저로 result.json 다운로드
except Exception as e:
    print("(자동 다운로드 불가 — Drive에서 직접 받으세요)", e)

---

## 다음에 또 끊길 때를 위해

이 노트북은 **아무 실험에나 쓸 수 있습니다.** `evaluation.json`만 있으면 복구됩니다.

**끊김에 더 강해지는 법** — 실험 노트북 CELL 2의 `RUN_ID`에 끊긴 run 폴더명을 넣고
다시 실행하면, **이미 끝난 arm은 `probs.npy`를 재사용**하고 남은 arm만 학습합니다.
코호트 캐시도 `data/`에 공유되므로 다운로드도 다시 하지 않습니다.

```python
RUN_ID = "20260731T1655_exp1p_icentia"   # ← 이어서 돌릴 폴더명
```
